# ***Phase 2: Model C - BERTopic Semantic Discovery***

This notebook focuses on training the **BERTopic** model to discover semantic topic clusters within the news corpus. This approach goes beyond the probabilistic LDA model by leveraging contextual embeddings (BERT). *italicised text*

### ***Goals***:
1. *Train BERTopic on processed news text*.
2. *Extract document-topic distributions*.
3. *Save the topic proportions for temporal forecasting in Phase 3.*

In [10]:
%pip install bertopic

In [11]:
import pandas as pd
import numpy as np
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
import os

# 1. Load Data
DATA_PATH = '/content/processed_featured_data.csv'
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df)} documents.")

# Handle potential NaN values
df['clean_text'] = df['clean_text'].fillna("")
docs = df['clean_text'].tolist()

Loaded 10000 documents.


In [12]:
# 2. Component Configuration
# UMAP for dimensionality reduction
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)

# HDBSCAN for clustering
hdbscan_model = HDBSCAN(min_cluster_size=10, metric='euclidean', cluster_selection_method='eom', prediction_data=True)

# Sentence Transformer for embeddings
sentence_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
# 3. Train BERTopic Model
# We limit the number of topics to around 20-30 for better forecasting interpretability
topic_model = BERTopic(
    embedding_model=sentence_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    nr_topics=30,
    calculate_probabilities=True,
    verbose=True
)

topics, probs = topic_model.fit_transform(docs)
print("Training completed.")

2026-04-18 06:34:12,680 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/313 [00:00<?, ?it/s]

2026-04-18 06:34:22,088 - BERTopic - Embedding - Completed ✓
2026-04-18 06:34:22,090 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-18 06:34:47,341 - BERTopic - Dimensionality - Completed ✓
2026-04-18 06:34:47,343 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-18 06:34:57,653 - BERTopic - Cluster - Completed ✓
2026-04-18 06:34:57,654 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-04-18 06:34:57,999 - BERTopic - Representation - Completed ✓
2026-04-18 06:34:58,000 - BERTopic - Topic reduction - Reducing number of topics
2026-04-18 06:34:58,026 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-18 06:34:58,352 - BERTopic - Representation - Completed ✓
2026-04-18 06:34:58,358 - BERTopic - Topic reduction - Reduced number of topics from 166 to 30


Training completed.


In [14]:
import os

# Ensure directory exists
os.makedirs('../data/processed', exist_ok=True)

# Store topic results
df['topic'] = topics

# Save outputs
np.save('../data/processed/topic_probabilities.npy', probs)
df.to_parquet('../data/processed/featured_news_with_topics.parquet', index=False)

print("Results saved to data/processed/")

Results saved to data/processed/
